# Gimlet Heterogeneous Precision/Width — Colab Runbook

**Goal:** Train for 600 steps on a T4 GPU, save `final_model.pt` + `final_model.ptz`, and compute BPB.

The training script (`train_gpt.py`) has **built-in T4 detection** — no manual regex patching needed.

| Cell | Purpose | Est. Time |
|------|---------|-----------|
| 1 | Setup: clone, deps, data | 3-5 min |
| 2 | Train 600 steps | 30-50 min |
| 3 | Verify artifacts + BPB | 2 min |

In [ ]:
# ====== CELL 1: SETUP ======
import os, subprocess, time

# 1a. Clone repos
%cd /content
if os.path.exists('pg'):
    !rm -rf pg
!git clone -b feat/gimlet-hetero-v1 https://github.com/jmoncayo-pursuit/parameter-golf-gimlet-hetero.git pg

if not os.path.exists('openai-pg'):
    !git clone https://github.com/openai/parameter-golf.git openai-pg

# 1b. Install deps
!pip install -q torch numpy sentencepiece zstandard

# 1c. Fetch data (tokenizer + 1 validation shard + 1 train shard)
DATA_DIR = '/content/openai-pg/data/datasets/fineweb10B_sp1024'
TOK_DIR  = '/content/openai-pg/data/tokenizers'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(TOK_DIR, exist_ok=True)

BASE_URL = 'https://openaipublic.blob.core.windows.net/parameter-golf'

def fetch(url, dest):
    if not os.path.exists(dest):
        print(f'Downloading {os.path.basename(dest)}...')
        !wget -q -O "{dest}" "{url}"
    else:
        print(f'{os.path.basename(dest)} already exists.')

fetch(f'{BASE_URL}/tokenizers/fineweb_1024_bpe.model',
      f'{TOK_DIR}/fineweb_1024_bpe.model')
fetch(f'{BASE_URL}/datasets/fineweb10B_sp1024/fineweb_val_000000.bin',
      f'{DATA_DIR}/fineweb_val_000000.bin')
fetch(f'{BASE_URL}/datasets/fineweb10B_sp1024/fineweb_train_000001.bin',
      f'{DATA_DIR}/fineweb_train_000001.bin')

# 1d. Create results directory
os.makedirs('/content/pg/results/gimlet_hetero_staged', exist_ok=True)

# 1e. Verify GPU is available
import torch
if torch.cuda.is_available():
    print(f'\n✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('❌ No GPU detected — go to Runtime → Change runtime type → GPU')

print('\n✅ Setup complete.')

In [ ]:
# ====== CELL 2: TRAIN 600 STEPS ======
import os, subprocess, time

%cd /content/pg

# Configure training
env = os.environ.copy()
env['ITERATIONS'] = '600'
env['TRAIN_SEQ_LEN'] = '512'         # Shorter sequences for T4 memory
env['TRAIN_BATCH_TOKENS'] = '16384'   # Smaller batch for T4
env['VAL_LOSS_EVERY'] = '50'          # Validate every 50 steps

print(f'Starting training at {time.strftime("%H:%M:%S")}...')
print(f'Config: ITERATIONS=600 SEQ_LEN=512 BATCH=16384')
print(f'---')

proc = subprocess.Popen(
    ['python3', 'train_gpt.py'],
    cwd='/content/pg',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream output in real time
for line in proc.stdout:
    print(line, end='', flush=True)

rc = proc.wait()
print(f'\n--- Training finished with exit code {rc} at {time.strftime("%H:%M:%S")} ---')

# Check for artifacts
for name in ['final_model.pt', 'final_model.ptz']:
    path = f'/content/pg/{name}'
    if os.path.exists(path):
        print(f'✅ {name}: {os.path.getsize(path)/1e6:.2f} MB')
    else:
        print(f'❌ {name} NOT FOUND')

In [ ]:
# ====== CELL 3: VERIFY ARTIFACTS + BPB ======
import os, sys, math, pathlib, zstandard, torch
import sentencepiece as spm
import importlib.util

def run_verification():
    # 1. Find checkpoint
    print('--- 1. CHECKPOINT SEARCH ---')
    ckpt = pathlib.Path('/content/pg/final_model.pt')
    if not ckpt.exists():
        all_pts = sorted(pathlib.Path('/content').rglob('*.pt'), key=lambda x: x.stat().st_mtime)
        if not all_pts:
            print('❌ No .pt files found. Run Cell 2 first.')
            return
        ckpt = all_pts[-1]
    print(f'Using: {ckpt} ({ckpt.stat().st_size/1e6:.2f} MB)')

    # 2. Compressed size check
    print('\n--- 2. ARTIFACT SIZE ---')
    raw = ckpt.read_bytes()
    cctx = zstandard.ZstdCompressor(level=10)
    compressed = cctx.compress(raw)
    comp_mb = len(compressed) / 1e6
    code_size = pathlib.Path('/content/pg/train_gpt.py').stat().st_size
    total = len(compressed) + code_size
    print(f'Raw: {len(raw)/1e6:.2f} MB')
    print(f'Compressed (zstd-10): {comp_mb:.2f} MB')
    print(f'Code: {code_size/1e3:.1f} KB')
    print(f'Total artifact: {total/1e6:.2f} MB')
    print(f'16 MB limit: {"PASS ✅" if total <= 16_000_000 else "FAIL ❌"}')

    # 3. BPB calculation
    print('\n--- 3. SLIDING-WINDOW BPB ---')
    sys.path.insert(0, '/content/pg')
    spec = importlib.util.spec_from_file_location('tg', '/content/pg/train_gpt.py')
    tg = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tg)

    device = torch.device('cuda')
    args = tg.Hyperparameters()
    sp = spm.SentencePieceProcessor(model_file='/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model')

    model = tg.GPT(
        vocab_size=args.vocab_size,
        num_layers=args.num_layers,
        model_dim=args.model_dim,
        num_heads=args.num_heads,
        num_kv_heads=args.num_kv_heads,
        tie_embeddings=args.tie_embeddings,
        tied_embed_init_std=args.tied_embed_init_std,
        logit_softcap=args.logit_softcap,
        rope_base=args.rope_base,
        qk_gain_init=args.qk_gain_init
    ).to(device).bfloat16()

    model.load_state_dict(torch.load(str(ckpt), map_location='cpu'))
    model.eval()

    seq_len = 512
    val_full = tg.load_validation_tokens(
        '/content/openai-pg/data/datasets/fineweb10B_sp1024/fineweb_val_*.bin', seq_len)
    mini_seqs = 128
    val_tokens = val_full[:mini_seqs * seq_len + 1].to(device, dtype=torch.int64)
    luts = tg.build_sentencepiece_luts(sp, args.vocab_size, device)

    x_all = val_tokens[:-1].reshape(-1, seq_len)
    y_all = val_tokens[1:].reshape(-1, seq_len)
    loss_sum, tok_count, byte_count = 0.0, 0, 0.0

    with torch.inference_mode():
        for i in range(x_all.size(0)):
            x, y = x_all[i:i+1], y_all[i:i+1]
            loss_sum += float(model(x, y).item()) * y.numel()
            tok_count += y.numel()
            tb = luts[0][y.reshape(-1)].to(torch.int16)
            tb += (luts[1][y.reshape(-1)] & ~luts[2][x.reshape(-1)]).to(torch.int16)
            byte_count += float(tb.sum().item())

    bpb = (loss_sum / tok_count / math.log(2.0)) * (tok_count / byte_count)

    print(f'\n=== FINAL RESULTS ===')
    print(f'BPB (mini-val 128 seqs): {bpb:.6f}')
    print(f'Artifact size: {total/1e6:.2f} MB / 16.00 MB')
    print(f'Status: {"READY FOR PR ✅" if total <= 16_000_000 else "OVER SIZE LIMIT ❌"}')

run_verification()